# The insult wall — what the accounts actually said, and who they said it to

Counting attacks tells you how *often* an account goes after someone (that's `corpus_eda`). This is the other half: the insults themselves — every quotable put-down the pipeline pulled out, tagged with who said it, who it hit, and what kind of insult it is.

A few terms:
- a **dunk line** is a caption-sized, quotable put-down the model lifted from a post.
- **own voice** means the account's own caption or on-screen text — not a song lyric, not a line from a clip it aired. Only own-voice lines count toward a tally; otherwise an account gets blamed for words it merely replayed.
- **target** is whoever the line hits, tagged line by line.

## Setup

In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)       # show every row — no truncated lists
pd.set_option("display.max_colwidth", None)   # show full cell text
from pathlib import Path

# portable data root
ROOT = Path.cwd()
for cand in (ROOT, *ROOT.parents):
    if (cand / "data" / "analysis" / "dunk_lines.csv").exists():
        ROOT = cand; break
AN = ROOT / "data" / "analysis"

dunks = pd.read_csv(AN / "dunk_lines.csv")
OWN = ["caption", "overlay", "meme_text"]
dunks["own_voice"] = dunks["source"].isin(OWN)

print(f"{len(dunks)} extracted dunk lines")
print("by account:", dunks["account"].value_counts().to_dict())
print("by voice  :", dunks["source"].value_counts().to_dict())
dunks.head(30)

3586 extracted dunk lines
by account: {'democrats': 2704, 'whitehouse': 607, 'republicans': 275}
by voice  : {'overlay': 1786, 'speech': 748, 'caption': 665, 'sound': 387}


,account,date,dunk_line,punch,target,source,views,video_id,own_voice
0,democrats,20220719,Marjorie Taylor Greene is one of the 192 House Republicans.,up (powerful figure),Marjorie Taylor Greene,caption,204200,7121926917104176430,True
1,democrats,20220719,192 House Republicans voted AGAINST funding to address the baby formula shortage.,lateral (institution),House Republicans,overlay,204200,7121926917104176430,True
2,democrats,20220719,"Hmm, funny, yes, but not funny, haha, funny, weird.",lateral (institution),Marjorie Taylor Greene; House Republicans,speech,204200,7121926917104176430,False
3,democrats,20220811,Republicans upset that the ✨Inflation Reduction Act✨ makes corporations pay their fair share in taxes,lateral (institution),Republican Party,overlay,13400,7130405513526840619,True
4,democrats,20220811,How could you do this to me? Question mark.,lateral (institution),Republican Party,sound,13400,7130405513526840619,False
5,democrats,20220816,These Republicans sided with Big Pharma,lateral (institution),Republican Party,caption,31400,7132505762009550126,True
6,democrats,20220816,We do not care.,lateral (institution),Republican Party,sound,31400,7132505762009550126,False
7,democrats,20250528,What you are about to witness will disturb you. Even shock you.,up (powerful figure),"Donald Trump, Kristi Noem, Markwayne Mullin, Elon Musk, J.D. Vance, Melania Trump, Clarence Thomas",overlay,1100000,7509614479517273390,True
8,democrats,20250528,There is a dark side of humanity the censors won't let you see... but we will.,up (powerful figure),"Donald Trump, Kristi Noem, Markwayne Mullin, Elon Musk, J.D. Vance, Melania Trump, Clarence Thomas",overlay,1100000,7509614479517273390,True
9,democrats,20250528,𝓲𝓽 𝓪𝓵𝓵 𝓶𝓪𝓴𝓮𝓼 𝓼𝓮𝓷𝓼𝓮 𝓷𝓸𝔀 . . .,up (powerful figure),"Donald Trump, Kristi Noem, Markwayne Mullin, Elon Musk, J.D. Vance, Melania Trump, Clarence Thomas",caption,1100000,7509614479517273390,True


## §1 — Whose words are these?

A line only counts as an account's insult if the account actually wrote it — its own caption or on-screen text. Anything spoken inside a clip it aired, or sung in a track it played, is borrowed, and I set it aside. It's the most important filter here: skip it and an account gets credited with everything it ever reposted.

In [2]:

voice = (dunks.groupby("account")
              .agg(extracted=("dunk_line", "size"), own_voice=("own_voice", "sum")))
voice["own_voice_%"] = (voice["own_voice"] / voice["extracted"] * 100).round(0)
voice.loc[["democrats", "republicans", "whitehouse"]]


,extracted,own_voice,own_voice_%
account,,,
democrats,2704,1952,72.0
republicans,275,165,60.0
whitehouse,607,334,55.0


## §2 — Who each account goes after

The DNC's main character is Trump; the RNC's is the Democratic Party. Target is tagged per own-voice
line, and I show names exactly as the classifier recorded them — near-duplicates like "Democratic
Party" and "the Democratic Party" stay separate, so you see the raw count rather than my tidying.

In [3]:

own = dunks[dunks["own_voice"]]
for acct in ["democrats", "republicans", "whitehouse"]:
    print(f"{acct} — most-targeted (own voice):")
    print(own[own["account"] == acct]["target"].value_counts().head(6).to_string())
    print()


democrats — most-targeted (own voice):
target
Donald Trump             987
Republican Party          84
JD Vance                  84
Kevin McCarthy            45
Robert F. Kennedy Jr.     35
Mike Johnson              29

republicans — most-targeted (own voice):
target
Democratic Party       64
James Talarico         10
Joe Biden               5
Gavin Newsom            5
Political opponents     4
Chuck Schumer           4

whitehouse — most-targeted (own voice):
target
Democratic Party       65
the media              18
Joe Biden              13
Nicolás Maduro         13
Political opponents    11
Iran                   10



## §3 — What kind of insult? The @democrats vocabulary by register

The vocabulary is the hand-curated lexicon — pulled from the vetted walls, not from a keyword scan — grouped by the register the account reaches for: **body** (looks, decay), **mind** (competence), **morals** (character), and a Vance-only **subservience** set. The split is the finding: the generic competence and character words are shared around, but the body-horror register is the DNC account's own signature.

In [4]:

lex = pd.read_csv(AN / "curated_insults.csv")
(lex.groupby("register")["term"]
    .agg(n="size", terms=lambda s: ", ".join(sorted(s)))
    .reindex(["body", "mind", "morals", "subservience"]))


,n,terms
register,,
body,22,"bad hair, bad posture, creature, crusty, decaying, decaying sack of flesh, decrepit, fat, fatty, gross, mouth breather, nasty, repulsive, scaly, snorlax, sweaty, tan, tiny, too many chins, ugly, wig, wrinkled"
mind,31,"80-year-old man, can't read, chopped, confused, cooked, crashing out, deteriorating, dumb, grandpa, insane, insecure, low energy, mad, melts down, mess, mogged, old, old as fart, oldest, rambling, sad sack, senile, sleepy, snoozing, stupid, unc, unhinged, unstable, unwell, useless, weird"
morals,33,"chicken, chud, convicted felon, diabolical, disgusting, epstein's bff, evil, fat chud, fat little chud, flop, haunted, horny, jealous, jobless, liar, loony bin, looter, loser, nightmare, pigging out, predator, rat, resident evil, sleep-paralysis demon, snowflake, spooky, tacky, tacky gold, traitor, true chud, vile, wannabe dictator, worst"
subservience,14,"bootlicker, concerning, creep, disgusting, dumb, irrelevant, jd-chan, mad, negative aura, owned, poor, sellout, ugly bootlicker, weird"


## §4 — The canonical walls, vetted by hand

`walls_print.md` is the locked inventory — every insult @democrats aimed at Trump and Vance, and every curse word on its feed, all checked line by line against the video. Here the three walls sit in full, one row per line, with a **context column**: the source post's link (`url`), its view count, and where the line appears (caption / overlay / speech), so each line is checkable at a glance. About four in five lines match back to a specific post; the rest are older or lightly paraphrased and show blank. Filter by `target` (`trump` / `vance` / `profanity`) to see one wall at a time.

In [5]:
import re

# parse all three vetted walls (Trump, Vance, curse words) from walls_print.md
text = (ROOT / "data" / "walls_print.md").read_text()
sections = {"Trump, according to Dems": "trump",
            "Vance, according to Dems": "vance",
            "Curse words, according to Dems": "profanity"}
rows = []
for section in text.split("## ")[1:]:
    head = section.split("\n", 1)[0].strip()
    if head not in sections:
        continue
    for line in section.splitlines():
        m = re.match(r"\s*(20\d\d)(?:\s*\(through July\))?\s+(.*)", line)
        if not m:
            continue
        year, body = m.group(1), m.group(2)
        for snippet in re.split(r"\s{2,}", body):     # wall lines are separated by 2+ spaces
            snippet = snippet.strip(" *")
            if len(snippet) > 1 and snippet != "—":
                rows.append({"target": sections[head], "year": year, "line": snippet})
walls = pd.DataFrame(rows)

# context: match each wall line back to its source post in dunk_lines
dem = dunks[dunks["account"] == "democrats"].copy()
normalize = lambda s: re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()
dem["key"] = dem["dunk_line"].map(normalize)
lookup = (dem.dropna(subset=["key"]).drop_duplicates("key")
             .set_index("key")[["video_id", "views", "source"]])

def find_post(line):
    key = normalize(line)
    if key in lookup.index:
        r = lookup.loc[key]; return r["video_id"], r["views"], r["source"]
    for k in lookup.index:                        # fall back to a close partial match
        if key and (key in k or k in key) and abs(len(key) - len(k)) < 25:
            r = lookup.loc[k]; return r["video_id"], r["views"], r["source"]
    return None, None, None

vid, views, src = zip(*walls["line"].map(find_post))
walls["source"] = src
walls["views"]  = views
walls["url"]    = [f"https://www.tiktok.com/@democrats/video/{int(v)}" if pd.notna(v) else ""
                   for v in vid]

print("wall lines:", walls.groupby("target").size().to_dict(),
      "| linked to a post:", int((walls["url"] != "").sum()), "of", len(walls))
walls

wall lines: {'profanity': 147, 'trump': 264, 'vance': 26} | linked to a post: 344 of 437


,target,year,line,source,views,url
0,trump,2023,60 seconds of Donald Trump blabbing... 🤴🦞💨🐋🤡,NaN,NaN,
1,trump,2023,we're glad you lost,overlay,1800000.0,https://www.tiktok.com/@democrats/video/7290984402627087646
2,trump,2023,Flop.,caption,167700.0,https://www.tiktok.com/@democrats/video/7236906798060588334
3,trump,2023,THUNDA,caption,1100000.0,https://www.tiktok.com/@democrats/video/7509614479517273390
4,trump,2023,Is Donald Trump... okay?🤡,caption,57200.0,https://www.tiktok.com/@democrats/video/7296932935758826798
5,trump,2024,A deeply confused Trump confuses Nancy Pelosi and Nikki Haley multiple times,NaN,NaN,
6,trump,2024,oh brother this guy stinks,sound,6638.0,https://www.tiktok.com/@democrats/video/7223775443105615147
7,trump,2024,Them being weird,overlay,152200.0,https://www.tiktok.com/@democrats/video/7402987614078422318
8,trump,2024,"Republicans Are Weird... These are weird people on the other side. Listen to them speak. These are weird ideas. [Trump moans] These are weird, weird people. Don't get sugarcoating.",NaN,NaN,
9,trump,2024,every presidential candidate has their strange wannabe,NaN,NaN,
